# Structured Lattice Experiment on Kaggle

Блокнот запускает пакетный эксперимент `em3d.experiments.structured_lattice` после установки пакета напрямую из GitHub. Для smoke-проверки уменьшите `N` и `MAX_ITER`; для рабочего запуска используйте `N=(100,100,100)` и `COARSE_N=(9,9,9)`.

In [ ]:
!pip install "em3d[vis] @ git+https://github.com/qwerty29544/em3d.git"


In [ ]:
from pathlib import Path
import json
import zipfile

from em3d.experiments.structured_lattice import (
    MaterialSpec,
    make_structured_lattice_case,
    run_structured_lattice_experiment,
)


## Параметры запуска

Для полного эксперимента оставьте значения ниже. Для быстрой проверки на Kaggle можно временно поставить `N=(24,24,24)`, `COARSE_N=(4,4,4)`, `LATTICE_SHAPE=(3,3,3)` и `MAX_ITER=20`.

In [ ]:
N = (100, 100, 100)
COARSE_N = (9, 9, 9)
LATTICE_SHAPE = (5, 5, 5)
INCLUSION_RADIUS = (0.045, 0.045, 0.045)
K0 = 8.0
MAX_ITER = 500
RTOL = 1e-6
RCS_N_PHI = 120

OUTPUT_ROOT = Path("/kaggle/working/structured-lattice-n100")


In [ ]:
case = make_structured_lattice_case(
    N=N,
    coarse_N=COARSE_N,
    lattice_shape=LATTICE_SHAPE,
    inclusion_radius=INCLUSION_RADIUS,
    material=MaterialSpec.isotropic(2.5 + 0.02j),
    k0=K0,
    solver_names=("SIM", "BiCGStab", "TwoStep"),
)

case


In [ ]:
summary = run_structured_lattice_experiment(
    case=case,
    output_root=OUTPUT_ROOT,
    max_iter=MAX_ITER,
    rtol=RTOL,
    rcs_n_phi=RCS_N_PHI,
    make_plots=True,
)

summary


In [ ]:
summary_path = OUTPUT_ROOT / "raw" / "structured_lattice_summary.json"
with summary_path.open("r", encoding="utf-8") as f:
    saved_summary = json.load(f)
saved_summary


In [ ]:
zip_path = Path("/kaggle/working/structured-lattice-results.zip")
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for path in OUTPUT_ROOT.rglob("*"):
        if path.is_file():
            zf.write(path, path.relative_to(OUTPUT_ROOT.parent))

zip_path
